# 11_window_features_CHS

- 목적: 전처리된 샘플 시계열에서 1·2·3초 윈도우 피처(진폭·형상·전류 사인 잔차)를 만들어 모델링 입력 계약(parquet)으로 고정하고, 공통 평가 함수로 규칙 베이스라인을 재현한다(착수 게이트).
- 입력: `data/raw`(없으면 zip 자동 추출) → `preprocess.preprocess()`, `data/processed/segments.csv`
- 출력: `figures/11_window_features_CHS/`, `results/11_window_features_CHS/`, `data/processed/11_window_features.parquet`
- 담당: CHS
- 심사 대응: 2번(모델 입력·평가 기준 고정), 6번(재현성), 5번(형상·사인 잔차 피처군)

파일명 규칙: 0x 진단 · 1x 전처리·피처 · 2x 모델 · 3x 분석

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "11_window_features_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
import data_quality as dq, preprocess, features, split, evaluate, segments
PQ = paths.DATA_PROCESSED / "11_window_features.parquet"
print(NB, FIG.relative_to(ROOT), RES.relative_to(ROOT))

11_window_features_CHS figures\11_window_features_CHS results\11_window_features_CHS


## 2. 로드 · 표준 전처리 (`preprocess.preprocess()` 기본 인자)

In [2]:
raw = dq.load_all()
df = pd.concat(raw.values(), ignore_index=True)
df_pre = preprocess.preprocess(df)
print(df_pre.shape, df_pre["src"].value_counts().to_dict(), "세그먼트", df_pre["seg_uid"].nunique())

(20600, 10) {'normal': 20000, 'outlier': 600} 세그먼트 620


## 3. 윈도우 피처 (1·2·3초, step 0.5초) → parquet

전류 사인 피팅은 세그먼트 단위 1회 계산 후 윈도우에 브로드캐스트한다.

In [3]:
W = features.window_features(df_pre)
W.to_parquet(PQ, index=False)
print(W.shape, "피처 열", len(features.feature_columns(W)))
print(W.groupby(["win_s", "label"]).size().unstack().rename(columns={0: "정상", 1: "이상"}))

(7115, 40) 피처 열 33
label    정상  이상
win_s          
1.0    3264  96
2.0    2254  63
3.0    1400  38


## 4. 커버리지 — 판정 불가(길이<윈도우) 세그먼트 비율

In [4]:
cov = features.segment_coverage(df_pre)
cov.to_csv(RES / "effective_samples.csv", index=False)
print(cov.to_string(index=False))
print(cov.pivot(index="win_s", columns="src", values="n_windows"))

 win_s     src  n_seg  n_seg_valid  n_seg_undetermined  undetermined_ratio  n_windows
   1.0  normal    599          530                  69              0.1152       3264
   1.0 outlier     21           17                   4              0.1905         96
   2.0  normal    599          452                 147              0.2454       2254
   2.0 outlier     21           13                   8              0.3810         63
   3.0  normal    599          360                 239              0.3990       1400
   3.0 outlier     21           10                  11              0.5238         38
src    normal  outlier
win_s                 
1.0      3264       96
2.0      2254       63
3.0      1400       38


## 5. 윈도우 단위 단독 피처 AUC

AUC는 방향 무시(`max(auc, 1-auc)`, 02 §7과 같은 정의). `n_outlier_seg`는 그 윈도우 길이에 참여하는 이상 세그먼트 수(선택 효과). `ref_auc_rms`는 02 §7 방식(원시 값, 샘플 stride 1 이동 RMS, 전처리 없음)의 참고값이다.

In [5]:
fcols = features.feature_columns(W)
rows = []
for ws, g in W.groupby("win_s"):
    y = g["label"].to_numpy()
    nseg_o = g.loc[g.label == 1, "seg_uid"].nunique()
    for c in fcols:
        a = evaluate.auc(y, g[c].fillna(g[c].median()))
        rows.append({"win_s": ws, "feature": c, "auc": round(max(a, 1 - a), 4), "n_normal": int((y == 0).sum()),
                     "n_outlier": int((y == 1).sum()), "n_outlier_seg": nseg_o})
A = pd.DataFrame(rows)
ref = segments.window_auc_table(raw["normal"], raw["outlier"], windows_sec=(1, 2, 3))
ref["feature"] = ref["channel"] + "_rms"; ref["win_s"] = ref["window_sec"].astype(float)
A = A.merge(ref[["win_s", "feature", "auc"]].rename(columns={"auc": "ref_auc_rms"}), on=["win_s", "feature"], how="left")
A.to_csv(RES / "window_feature_auc.csv", index=False)
print(A.sort_values(["win_s", "auc"], ascending=[True, False]).groupby("win_s").head(8).to_string(index=False))
print(A[A.feature.str.endswith("_rms")].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True)
for ax, (ws, g) in zip(axes, A.groupby("win_s")):
    t = g.sort_values("auc").tail(10)
    ax.barh(t["feature"], t["auc"]); ax.set_title(f"윈도우 {ws:g}초 단독 피처 AUC 상위 10"); ax.set_xlim(0.5, 1.0)
paths.save_fig(FIG, "window_feature_auc.png")

 win_s                  feature    auc  n_normal  n_outlier  n_outlier_seg  ref_auc_rms
   1.0        cur_fit_resid_rms 1.0000      3264         96             17          NaN
   1.0               cur_fit_r2 1.0000      3264         96             17          NaN
   1.0            cur_fit_f_dev 1.0000      3264         96             17          NaN
   1.0                  cur_ac1 0.9987      3264         96             17          NaN
   1.0               vib_corr01 0.9245      3264         96             17          NaN
   1.0        AI2_Current_crest 0.9010      3264         96             17          NaN
   1.0       AI1_Vibration_skew 0.7914      3264         96             17          NaN
   1.0 AI1_Vibration_shape_skew 0.7914      3264         96             17          NaN
   2.0        cur_fit_resid_rms 1.0000      2254         63             13          NaN
   2.0               cur_fit_r2 1.0000      2254         63             13          NaN
   2.0            cur_fit_f_dev 

saved figures\11_window_features_CHS\window_feature_auc.png


## 6. 분할 열 부여 (`fold`, `block`, seed=42)

분할은 **윈도우 길이와 무관하게 전체 620 세그먼트 표(`segments.csv`) 기준**으로 한 번 정해 모든 윈도우 길이에 같은 열을 쓴다. `block`은 정상만(시간순 5블록), 이상은 NaN. 세그먼트 메타(`state`, `vib_grade`, `cur_grade`)도 함께 붙인다.

In [6]:
S = pd.read_csv(paths.DATA_PROCESSED / "segments.csv", encoding="utf-8-sig")
S["start"] = pd.to_datetime(S["start"])
assert set(W["seg_uid"]) <= set(S["seg_uid"])
fold = split.fold_assignment(S, n_splits=5, seed=42)
block = split.block_assignment(S[S.label == 0], n_blocks=5)
W = W.drop(columns=[c for c in ["fold", "block", "state", "vib_grade", "cur_grade"] if c in W.columns])
W["fold"] = W["seg_uid"].map(fold).astype(int)
W["block"] = W["seg_uid"].map(block).astype("Int64")
W = W.merge(S[["seg_uid", "state", "vib_grade", "cur_grade"]], on="seg_uid", how="left")
assert (S.set_index("seg_uid").loc[W["seg_uid"], "label"].to_numpy() == W["label"].to_numpy()).all()
W.to_parquet(PQ, index=False)
print(split.split_summary(S, fold).to_string())
print(split.split_summary(S[S.label == 0], block).to_string())
print("parquet", PQ.relative_to(ROOT), W.shape)

      n_segments  n_outlier_seg  n_rows               start_min               start_max  share_state0.0  share_state1.0
fold                                                                                                                   
0            125              5     125 2022-07-12 00:00:07.224 2022-07-17 10:53:42.668          0.3750          0.6250
1            124              4     124 2022-07-12 00:00:15.267 2022-07-17 10:53:52.140          0.4250          0.5750
2            124              4     124 2022-07-12 00:00:32.752 2022-07-17 10:52:53.769          0.3833          0.6167
3            124              4     124 2022-07-12 00:00:47.676 2022-07-17 10:53:18.444          0.3917          0.6083
4            123              4     123 2022-07-12 00:00:00.019 2022-07-17 10:53:34.639          0.5294          0.4706
       n_segments  n_outlier_seg  n_rows               start_min               start_max  share_state0.0  share_state1.0
block                                  

## 7. 착수 게이트 — 규칙 베이스라인 재현

1초 윈도우 AI0 rms, 임계값 = train 정상 점수 q99(`threshold_from_normal`). 지연은 윈도우 완성 시점(첫 초과 t_start + 0.9초) 기준이며 체감 지연 = 윈도우 내 지연 + 7.958초. `group_kfold_seg`(fold 열)와 `time_block`(block 열, forward-chaining + 이상 전체 합침) 두 분할을 모두 보고한다.

In [7]:
W1 = pd.read_parquet(PQ).query("win_s == 1.0").reset_index(drop=True)
SCORE, MODEL, WIN_S = "AI0_Vibration_rms", "rule_rms_ai0_1s", 1.0

def eval_split(w, tr_mask, te_idx, split_name, fold_id):
    thr = evaluate.threshold_from_normal(w.loc[tr_mask & (w.label == 0), SCORE])
    te = w.loc[te_idx]; n, o = te[te.label == 0], te[te.label == 1]
    dl = evaluate.detection_delay_s(o[SCORE], o.seg_uid, o.t_start, thr, win_s=WIN_S)
    dl_gap = evaluate.detection_delay_s(o[SCORE], o.seg_uid, o.t_start, thr, win_s=WIN_S, include_gap=True)
    row = evaluate.to_metrics_row(MODEL, split_name, fold_id, evaluate.auc(te.label, te[SCORE]),
                                  evaluate.fpr_sample(n[SCORE], thr), evaluate.fpr_segment(n[SCORE], n.seg_uid, thr),
                                  dl["delay_s"].median())
    row.update({"thr": thr, "n_out_seg": len(dl), "n_detected": int(dl["detected"].sum()),
                "felt_delay_median_s": dl_gap["delay_s"].median()})
    return row

rows = []
for k in range(5):
    rows.append(eval_split(W1, W1.fold != k, np.flatnonzero(W1.fold == k), "group_kfold_seg", k))
for k in range(1, 5):
    te = split.append_outliers(np.flatnonzero((W1.block == k).fillna(False).to_numpy(bool)), W1)
    rows.append(eval_split(W1, (W1.block < k).fillna(False).to_numpy(bool), te, "time_block", k))
M = pd.DataFrame(rows)
mean_rows = []
for sp, g in M.groupby("split", sort=False):
    r = g.drop(columns=["model", "split", "fold"]).mean(numeric_only=True).to_dict()
    r.update({"model": MODEL, "split": sp, "fold": "mean"}); mean_rows.append(r)
M = pd.concat([M, pd.DataFrame(mean_rows)], ignore_index=True)
M["fold"] = M["fold"].astype(str)
M = M[evaluate.METRIC_COLS + ["thr", "n_out_seg", "n_detected", "felt_delay_median_s"]]
M.to_csv(RES / "metrics_smoke.csv", index=False)
print(M.round(4).to_string(index=False))
print("02 §7 참고: 1초 AI0 rms AUC 0.7588, 규칙 지연 중앙값 0.9초 / 체감 약 9.68초")

          model           split fold    auc  fpr_sample  fpr_segment  delay_median_s    thr  n_out_seg  n_detected  felt_delay_median_s
rule_rms_ai0_1s group_kfold_seg    0 0.8340      0.0212       0.0550            1.15 0.1255        5.0         4.0                9.108
rule_rms_ai0_1s group_kfold_seg    1 0.8488      0.0200       0.0769            0.90 0.1255        4.0         3.0                8.858
rule_rms_ai0_1s group_kfold_seg    2 0.9991      0.0016       0.0098            0.90 0.1299        2.0         2.0                8.858
rule_rms_ai0_1s group_kfold_seg    3 0.5724      0.0073       0.0273            0.90 0.1288        3.0         3.0                8.858
rule_rms_ai0_1s group_kfold_seg    4 0.6777      0.0046       0.0286            0.90 0.1291        3.0         2.0                8.858
rule_rms_ai0_1s      time_block    1 0.7615      0.0082       0.0098            0.90 0.1252       17.0        14.0                8.858
rule_rms_ai0_1s      time_block    2 0.7615     

## 8. 결론 — 팀원용 컬럼 계약

- 파일: `data/processed/11_window_features.parquet` (윈도우 1행, `win_s` ∈ {1, 2, 3}로 필터해 사용)
- **모델 입력 가능**: `features.feature_columns()`가 돌려주는 열 = `<채널>_{rms,peak,p2p,kurt,crest,skew}`(진폭 18), `<채널>_shape_{kurt,crest,skew}`(형상 9), `cur_fit_{resid_rms,r2,f_dev}`(사인 잔차 3), `vib_corr01`·`cur_ac1`·`vib_rms_ratio`(채널 관계 3) = 피처 열 33. `vib_rms_ratio`(윈도우 단위 AI0 rms / AI1 rms)만 윈도우 내용으로 계산하고, 나머지 관계 피처는 세그먼트 값 브로드캐스트
- **입력 금지**: `t_abs`(날짜=라벨, 시간 블록 정렬 전용), `src`, `label`(정답), `fold`, `block`, `seg_uid`(그룹 키), `t_start`, `n_samples`, `win_s`
- **사후 해석용**: `state`(정상만), `vib_grade`, `cur_grade`(이상만) — FN/FP 분석에만 사용
- 분할은 `fold`(GroupKFold 5) / `block`(정상 시간순 5블록, 이상은 NaN) 열을 그대로 쓴다. 윈도우 표에 `split.group_kfold_seg`를 다시 호출하지 않는다(윈도우 길이별로 fold가 달라짐).
- `cur_fit_*`은 세그먼트 전체 사인 피팅을 브로드캐스트한 값이라 윈도우 내용과 무관한 세그먼트 수준 피처다. 별도 ablation으로만 보고한다.
- 판정 불가 세그먼트(길이<윈도우)는 윈도우가 없어 평가에서 제외되므로 셀 4 비율을 결과와 함께 보고한다.